# PECT-JEPA cross_condition — Colab CPU runner // TAB 2b: exp57b (H6 sạch + scan-centered, nền exp57a)

Mỗi tab = 1 exp (`EXP_NAME` khác nhau). Chạy từ trên xuống dưới. Khi Colab ngắt, chạy lại cell resume (cell cuối) — train tiếp từ `latest_*.pt`, không mất.
Lưu ý: CPU chậm hơn GPU ~10–50x. Smoke trước (cell smoke), rồi mới chunk 10–15 epoch/lần.

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os, glob
DATA = '/content/drive/MyDrive/PECT_Dataset/'
DRIVE_RUNS = '/content/drive/MyDrive/PECT_Dataset/runs/'
os.makedirs(DRIVE_RUNS, exist_ok=True)
# Verify: TDMS + GT masks phải thấy từ Drive (lỗi cũ: train dùng data_dir='data' local -> 0 scans)
tdms = [f for f in glob.glob(os.path.join(DATA, '**', '*.tdms'), recursive=True) if not f.endswith('_index.tdms')]
print('TDMS:', len(tdms))
for p in [os.path.join(DATA, 'ground_truth/corrosion/corrosion_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v1/rivet_v1_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v2/rivet_v2_gt_mask.npy')]:
    print(('OK  ' if os.path.exists(p) else 'MISS') + ' ' + p)

TDMS: 81
OK   /content/drive/MyDrive/PECT_Dataset/ground_truth/corrosion/corrosion_gt_mask.npy
OK   /content/drive/MyDrive/PECT_Dataset/ground_truth/rivet_v1/rivet_v1_gt_mask.npy
OK   /content/drive/MyDrive/PECT_Dataset/ground_truth/rivet_v2/rivet_v2_gt_mask.npy


In [3]:
!test -d GraduationProject && (cd GraduationProject && git pull --ff-only) || git clone https://github.com/TranAnh35/GraduationProject.git
!cd GraduationProject && git switch Define_PECT_JEPA && git log --oneline -3

remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 15 (delta 12), reused 11 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (15/15), 3.83 KiB | 392.00 KiB/s, done.
From https://github.com/TranAnh35/GraduationProject
   97ce68b..e88e4e9  Define_PECT_JEPA -> origin/Define_PECT_JEPA
Updating 97ce68b..e88e4e9
Fast-forward
 .../cross_condition/colab/train_exam.ipynb         | 164 ++++++++++++++++++---
 .../cross_condition/evaluation/cscan_extractor.py  |   9 +-
 src/PECT_JEPA/cross_condition/latent_audit.py      |   5 +-
 3 files changed, 155 insertions(+), 23 deletions(-)
Already on 'Define_PECT_JEPA'
Your branch is up to date with 'origin/Define_PECT_JEPA'.
e88e4e9 (HEAD -> Define_PECT_JEPA, origin/Define_PECT_JEPA) Guard pin_memory/non_blocking on CUDA only (CPU Colab fix)
940fd1a Fix colab smoke: export EXP/BASE/EXTRA/DATA to environ so bash expands \ (IPython drops it)
97ce68b Fix c

In [4]:
!pip install -q uv nptdms scikit-learn scipy tqdm 2>&1 | tail -2
!cd GraduationProject && python -c "import torch, sklearn, scipy, nptdms; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())"

torch 2.11.0+cpu | cuda: False


In [5]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
# ===== CHỌN 1 EXP CHO TAB NÀY =====
EXP_NAME = 'exp57b_centered_scat_oracle'  # TAB 2b (H6 sach + scan-centered, diagnostic co GT)
BASE_FLAGS = '--star_radii 2 4 7 --tokenizer_type energy_adaptive --latent_disentanglement true --carrier_flat_weight 0.0 --scattering_var_weight 0.5 --inv_start_epoch 11 --inv_ramp_epochs 20 --xcov_start_epoch 11 --xcov_ramp_epochs 20 --inv_cosine true --zc_cond_weight 0.1 --var_gamma 0.5 --offset_weight 0.0 --pool_var_weight 0.0 --device cpu'
EXTRA_ALL = {
    'exp56_contpos_wedgemask': '--continuous_pos_embed true --contiguous_masking true --scattering_cov_weight 0.05 --inv_weight 0.5 --xcov_weight 0.05',
    'exp57a_centered_scat': '--continuous_pos_embed true --contiguous_masking true --predictor_scattering_only true --scan_centered_inv true --scattering_cov_weight 0.2 --inv_weight 0.2 --xcov_weight 0.05',
    'exp57b_centered_scat_oracle': '--continuous_pos_embed true --contiguous_masking true --predictor_scattering_only true --scan_centered_inv true --inv_oracle_mask true --scattering_cov_weight 0.2 --inv_weight 0.2 --xcov_weight 0.05',
    'exp58a_patch_energy_norm': '--patch_energy_norm true --scattering_cov_weight 0.05 --inv_weight 0.5 --xcov_weight 0.05',
}
EXTRA = EXTRA_ALL[EXP_NAME]  # tự theo EXP_NAME, mỗi tab chỉ đổi 1 dòng trên
# Export sang environ để cell shell (!) dùng ${VAR} chuẩn bash (IPython không expand ${VAR} từ Python namespace -> mất argument như lỗi smoke vừa gặp)
os.environ.update({'EXP_NAME': EXP_NAME, 'BASE_FLAGS': BASE_FLAGS, 'EXTRA': EXTRA, 'DATA': DATA, 'DRIVE_RUNS': DRIVE_RUNS})
print(EXP_NAME)
print(EXTRA)

exp56_contpos_wedgemask
--continuous_pos_embed true --contiguous_masking true --scattering_cov_weight 0.05 --inv_weight 0.5 --xcov_weight 0.05


In [6]:
# Smoke 1 epoch CPU (~vài phút): phải thấy 'Training C-scans' > 0 và CSV có metrics. Xong thì XÓA run smoke.
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --exp_name "${EXP_NAME}_smoke" --add_timestamp true $BASE_FLAGS $EXTRA --data_dir "$DATA" --epochs 1 --samples_per_epoch 256 --eval_val_every 1

[13:53:22 | INFO] Initialized Unified Experiment Directory: experiments/cross_condition/runs/_20261016_135322
[13:53:22 | INFO] Configuration: CrossConditionConfig(exp_name='', add_timestamp=True, base_exp_dir='experiments/cross_condition/runs', embed_dim=64, encoder_depth=4, encoder_heads=4, predictor_depth=2, predictor_heads=4, mlp_ratio=4.0, dropout=0.0, spatial_topology='concentric_star', star_radii=(2, 4, 7), grid_size=5, temporal_samples=128, num_harmonics=16, num_target_probes=9, num_context_probes=16, continuous_pos_embed=True, contiguous_masking=True, predictor_scattering_only=False, scan_centered_inv=False, patch_energy_norm=False, use_condition_token=True, condition_modulation='query_projection', tokenizer_type='energy_adaptive', relative_target=False, latent_disentanglement=True, carrier_dim=32, scattering_dim=32, carrier_flat_weight=0.0, scattering_var_weight=0.5, scattering_cov_weight=0.05, inv_weight=0.5, xcov_weight=0.05, inv_start_epoch=11, inv_ramp_epochs=20, xcov_sta

In [ ]:
# Chunk 1 (fresh): sửa --epochs theo chunk đầu (VD 15). Ghi nhớ run_dir in ra ở log 'Initialized Unified Experiment Directory'.
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --exp_name $EXP_NAME --add_timestamp true $BASE_FLAGS $EXTRA --data_dir "$DATA" --epochs 15 --eval_val_every 1

In [ ]:
# Backup run mới nhất ra Drive (chạy sau mỗi chunk, và trước khi Colab ngắt).
!RUN=$(ls -dt GraduationProject/experiments/cross_condition/runs/${EXP_NAME}_* | head -1); echo "$RUN"; mkdir -p "$DRIVE_RUNS"; cp -r "$RUN" "$DRIVE_RUNS/"; echo BACKUP_DONE

In [ ]:
# Resume chunk tiếp theo: GIỮ NGUYÊN toàn bộ flags, chỉ thêm --resume_checkpoint và NÂNG --epochs lên tổng tích lũy (VD 15 -> 30 -> 45 ... 100).
# Nếu Colab ngắt: mount lại Drive (cell 1), restore run về local rồi chạy cell này.
!test -d "GraduationProject/experiments/cross_condition/runs/" || echo RESTORE_NEEDED
!RUN=$(ls -dt GraduationProject/experiments/cross_condition/runs/${EXP_NAME}_* 2>/dev/null | head -1); if [ -z "$RUN" ]; then SRC=$(ls -dt ${DRIVE_RUNS}/${EXP_NAME}_* | head -1); echo "restore $SRC"; cp -r "$SRC" GraduationProject/experiments/cross_condition/runs/; fi; ls -dt GraduationProject/experiments/cross_condition/runs/${EXP_NAME}_* | head -3
!RUN=$(ls -dt GraduationProject/experiments/cross_condition/runs/${EXP_NAME}_* | head -1); CKPT="$RUN/checkpoints/latest_cross_condition_jepa.pt"; ls -la "$CKPT" && cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --exp_name $EXP_NAME $BASE_FLAGS $EXTRA --data_dir "$DATA" --epochs 30 --eval_val_every 1 --resume_checkpoint "$CKPT"

## Vì sao bản cũ crash
- Cell train không truyền `--data_dir` nên `train.py:44` mặc định `data/` local rỗng → `PairedPECTDataset` thấy 0 scans → `min() empty` (`paired_dataset.py:160`).
- `transform_csv` bị hardcode `data/...` (`train.py:167`) nhưng dataset/telemetry GT đều giải qua `data_dir` (`paired_dataset.py:86`, `train.py:202`) nên chỉ cần `--data_dir` trỏ Drive là đủ.
- Deps thiếu + run local mất khi ngắt mạng: đã thêm cell verify/deps/backup + resume tích lũy `--epochs` (`train.py:100-106`, `trainer.py:623`).